# Tahap 2: Case Representation
**CBR - Pidana Umum: Pemalsuan**

Tujuan: Representasikan setiap putusan dalam struktur data terorganisir.

**Langkah:**
1. Ekstraksi metadata (nomor perkara, tanggal, pasal, pihak)
2. Ekstraksi konten kunci (fakta, amar putusan, dakwaan)
3. Feature Engineering (word count, bag-of-words)
4. Simpan ke CSV dan JSON

## 1. Import Library

In [1]:
import os
import re
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# Path konfigurasi
RAW_DIR       = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load mapping case_id → filename asli
mapping_file = RAW_DIR / "case_mapping.json"
if mapping_file.exists():
    with open(mapping_file, encoding="utf-8") as f:
        case_mapping = {m["case_id"]: m["original_filename"] for m in json.load(f)}
else:
    case_mapping = {}

txt_files = sorted(RAW_DIR.glob("case_*.txt"))
print(f"✅ Ditemukan {len(txt_files)} file teks di {RAW_DIR}")
print(f"📁 Output akan disimpan di: {PROCESSED_DIR.resolve()}")

✅ Ditemukan 35 file teks di ..\data\raw
📁 Output akan disimpan di: E:\Semester 6\Penalaran Komputer\SubCPMK 3\Test_1\data\processed


## 2. Fungsi Ekstraksi Metadata

In [2]:
def extract_no_perkara(text: str) -> str:
    """
    Ekstrak nomor perkara dari teks putusan.
    Pola: 123/Pid.B/2023/PN.Xxx atau 456/Pid/2022/PT.Xxx
    """
    patterns = [
        r'(\d+/pid\.b/\d{4}/pn\.?[a-z]+)',
        r'(\d+/pid/\d{4}/p[nt]\.?[a-z]+)',
        r'(\d+/pid\.sus/\d{4}/pn\.?[a-z]+)',
        r'nomor[:\s]*(\d+[/\w.]+\d{4}[/\w.]+)',
        r'putusan[\s]*nomor[:\s]*(\d+[/\w.]+)',
        r'perkara[\s]*nomor[:\s]*(\d+[/\w.]+)',
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            return match.group(1).strip().upper()
    return "TIDAK_DITEMUKAN"


def extract_tanggal(text: str) -> str:
    """
    Ekstrak tanggal putusan.
    Pola: 10 Januari 2023, 5 februari 2022, dst.
    """
    bulan_map = {
        'januari':'01','februari':'02','maret':'03','april':'04',
        'mei':'05','juni':'06','juli':'07','agustus':'08',
        'september':'09','oktober':'10','november':'11','desember':'12'
    }
    # Cari pola tanggal diucapkan
    patterns = [
        r'(\d{1,2})\s+(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\s+(\d{4})',
        r'tanggal\s+(\d{1,2})\s+(\w+)\s+(\d{4})',
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            day, month_str, year = match.groups()
            month = bulan_map.get(month_str.lower(), '00')
            return f"{year}-{month}-{int(day):02d}"
    # Fallback: pola DD-MM-YYYY atau DD/MM/YYYY
    match = re.search(r'(\d{2})[/-](\d{2})[/-](\d{4})', text)
    if match:
        d, m, y = match.groups()
        return f"{y}-{m}-{d}"
    return "TIDAK_DITEMUKAN"


def extract_pasal(text: str) -> str:
    """
    Ekstrak pasal-pasal yang digunakan dalam dakwaan/putusan.
    Fokus pada pasal pemalsuan (Pasal 263-276 KUHP).
    """
    # Pasal pemalsuan KUHP: 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276
    patterns = [
        r'pasal\s+(\d+(?:\s*(?:dan|,|jo\.?)\s*\d+)*)\s+(?:kuhp|kitab|undang)',
        r'pasal\s+(2[6-7]\d)\s+kuhp',
        r'melanggar\s+pasal\s+(\d+)',
        r'pasal\s+(\d+)\s+ayat\s+\((\d+)\)',
    ]
    found = []
    for pat in patterns:
        matches = re.findall(pat, text, re.IGNORECASE)
        found.extend([m if isinstance(m, str) else ' '.join(m) for m in matches])
    
    # Unik dan bersihkan
    unique_pasal = list(dict.fromkeys([p.strip() for p in found if p.strip()]))
    return ', '.join(unique_pasal[:5]) if unique_pasal else "TIDAK_DITEMUKAN"


def extract_pihak(text: str) -> dict:
    """
    Ekstrak nama terdakwa dari putusan pidana.
    """
    terdakwa = "TIDAK_DITEMUKAN"
    
    patterns_terdakwa = [
        r'terdakwa[\s:]+([A-Z][A-Z\s]+?)(?:\s+(?:bin|binti|alias|als|,|;))',
        r'nama\s*lengkap\s*[:\s]+([A-Z][A-Z\s]{3,40})',
        r'terdakwa\s*:\s*([A-Z][\w\s]+)',
    ]
    for pat in patterns_terdakwa:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            terdakwa = match.group(1).strip().title()
            break
    
    return {"terdakwa": terdakwa, "penuntut": "Jaksa Penuntut Umum"}


def extract_pengadilan(text: str) -> str:
    """
    Ekstrak nama pengadilan dari putusan.
    """
    patterns = [
        r'pengadilan negeri\s+([\w\s]+?)(?:\s+yang|\s+dalam|\n|,)',
        r'p\.n\.?\s+([\w]+)',
        r'pn\.?([\w]+)',
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            return "Pengadilan Negeri " + match.group(1).strip().title()
    return "TIDAK_DITEMUKAN"


print("✅ Fungsi ekstraksi metadata siap")

✅ Fungsi ekstraksi metadata siap


## 3. Fungsi Ekstraksi Konten Kunci

In [3]:
def extract_dakwaan(text: str, max_len: int = 600) -> str:
    """
    Ekstrak bagian dakwaan dari teks putusan.
    """
    patterns = [
        r'dakwaan[\s\S]{0,50}?(?:kesatu|pertama|tunggal)[:\s]+([\s\S]{50,}?)(?=tuntutan|pembuktian|pertimbangan|\Z)',
        r'surat dakwaan[:\s]+([\s\S]{50,500})',
        r'didakwa[:\s]+([\s\S]{50,400})',
        r'dakwaan[:\s]+([\s\S]{50,400})',
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            snippet = match.group(1).strip()
            # Bersihkan whitespace berlebih
            snippet = re.sub(r'\s+', ' ', snippet)
            return snippet[:max_len]
    
    # Fallback: cari paragraf yang mengandung kata kunci dakwaan
    paragraphs = text.split('\n\n')
    for para in paragraphs:
        if any(kw in para.lower() for kw in ['memalsukan', 'pemalsuan', 'surat palsu', 'dokumen palsu']):
            return re.sub(r'\s+', ' ', para).strip()[:max_len]
    
    return "TIDAK_DITEMUKAN"


def extract_amar_putusan(text: str, max_len: int = 500) -> str:
    """
    Ekstrak amar putusan (bagian dispositif).
    Bagian terpenting sebagai 'solusi' dalam CBR.
    """
    patterns = [
        r'(?:mengadili|amar putusan)[\s\S]{0,30}?(?:menyatakan|memutuskan)[:\s]*([\s\S]{50,}?)(?=demikian|hakim|ketua|\Z)',
        r'memutuskan[:\s]*([\s\S]{50,600})',
        r'amar putusan[:\s]*([\s\S]{50,500})',
        r'mengadili[:\s]*([\s\S]{50,500})',
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            snippet = match.group(1).strip()
            snippet = re.sub(r'\s+', ' ', snippet)
            return snippet[:max_len]
    return "TIDAK_DITEMUKAN"


def extract_vonis(text: str) -> str:
    """
    Ekstrak vonis/hukuman dari amar putusan.
    """
    patterns = [
        r'(?:pidana penjara|hukuman penjara)[\s]*(?:selama)?[\s]*(\d+[\s]*(?:tahun|bulan|hari)[\s\w]*)',
        r'penjara[\s]*(\d+[\s]*(?:tahun|bulan))',
        r'(\d+\s*(?:tahun|bulan|hari))\s*penjara',
        r'bebas dari dakwaan',
        r'bebas dari segala tuntutan',
        r'lepas dari segala tuntutan',
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            return match.group(0).strip() if not match.groups() else match.group(1).strip()
    return "TIDAK_DITEMUKAN"


def extract_ringkasan_fakta(text: str, max_len: int = 500) -> str:
    """
    Ekstrak ringkasan fakta persidangan.
    """
    patterns = [
        r'(?:fakta[- ]fakta|fakta hukum|fakta persidangan)[:\s]+([\s\S]{50,}?)(?=pertimbangan|pembuktian|\Z)',
        r'(?:menimbang|mempertimbangkan)[\s,]+bahwa[:\s]+([\s\S]{50,400})',
        r'(?:keterangan saksi|saksi menerangkan)[:\s]+([\s\S]{50,400})',
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            snippet = match.group(1).strip()
            snippet = re.sub(r'\s+', ' ', snippet)
            return snippet[:max_len]
    return "TIDAK_DITEMUKAN"


print("✅ Fungsi ekstraksi konten kunci siap")

✅ Fungsi ekstraksi konten kunci siap


## 4. Feature Engineering

In [5]:
# Stopwords bahasa Indonesia sederhana
STOPWORDS_ID = set([
    'yang', 'dan', 'di', 'ke', 'dari', 'ini', 'itu', 'dengan', 'untuk',
    'pada', 'adalah', 'dalam', 'oleh', 'tidak', 'atau', 'juga', 'sudah',
    'saat', 'telah', 'bahwa', 'akan', 'ada', 'dapat', 'atas', 'kepada',
    'tersebut', 'sebagai', 'majelis', 'hakim', 'pengadilan', 'nomor',
    'tentang', 'antara', 'setelah', 'sehingga', 'karena', 'berdasarkan',
    'hal', 'maka', 'demikian', 'pula', 'yaitu', 'yakni', 'menurut'
])

def compute_features(text: str) -> dict:
    """
    Hitung fitur-fitur teks:
    - Jumlah kata
    - Jumlah kalimat
    - Top keywords
    - Kehadiran kata kunci pemalsuan
    """
    words = text.lower().split()
    sentences = re.split(r'[.!?]+', text)
    
    # Filter stopwords
    content_words = [w for w in words if w not in STOPWORDS_ID and len(w) > 3 and w.isalpha()]
    
    # Top 10 kata
    word_freq = Counter(content_words)
    top_keywords = ', '.join([w for w, _ in word_freq.most_common(10)])
    
    # Kata kunci spesifik pemalsuan
    pemalsuan_keywords = [
        'memalsukan', 'pemalsuan', 'palsu', 'surat palsu', 'dokumen palsu',
        'identitas palsu', 'stempel palsu', 'tanda tangan palsu',
        'keterangan palsu', 'ijazah palsu', 'merek palsu'
    ]
    found_pemalsuan = [kw for kw in pemalsuan_keywords if kw in text.lower()]
    
    return {
        "word_count": len(words),
        "sentence_count": len([s for s in sentences if s.strip()]),
        "unique_words": len(set(content_words)),
        "top_keywords": top_keywords,
        "pemalsuan_keywords": ', '.join(found_pemalsuan),
        "avg_sentence_length": round(len(words) / max(len(sentences), 1), 1)
    }

print("✅ Fungsi feature engineering siap")

✅ Fungsi feature engineering siap


## 5. Pipeline Utama: Representasi Semua Kasus

In [6]:
cases = []

for txt_file in tqdm(txt_files, desc="Memproses kasus"):
    case_id = txt_file.stem  # case_001, case_002, dst.
    text = txt_file.read_text(encoding="utf-8")
    
    # Ekstraksi metadata
    no_perkara  = extract_no_perkara(text)
    tanggal     = extract_tanggal(text)
    pasal       = extract_pasal(text)
    pihak       = extract_pihak(text)
    pengadilan  = extract_pengadilan(text)
    
    # Ekstraksi konten kunci
    dakwaan         = extract_dakwaan(text)
    amar_putusan    = extract_amar_putusan(text)
    vonis           = extract_vonis(text)
    ringkasan_fakta = extract_ringkasan_fakta(text)
    
    # Feature Engineering
    features = compute_features(text)
    
    # Gabungkan semua
    case_record = {
        # Identitas
        "case_id"            : case_id,
        "original_filename"  : case_mapping.get(case_id, f"{case_id}.pdf"),
        
        # Metadata
        "no_perkara"         : no_perkara,
        "tanggal"            : tanggal,
        "jenis_perkara"      : "Pidana Umum - Pemalsuan",
        "pasal"              : pasal,
        "terdakwa"           : pihak["terdakwa"],
        "penuntut"           : pihak["penuntut"],
        "pengadilan"         : pengadilan,
        
        # Konten Kunci
        "ringkasan_fakta"    : ringkasan_fakta,
        "dakwaan"            : dakwaan,
        "amar_putusan"       : amar_putusan,
        "vonis"              : vonis,
        
        # Features
        "word_count"         : features["word_count"],
        "sentence_count"     : features["sentence_count"],
        "unique_words"       : features["unique_words"],
        "top_keywords"       : features["top_keywords"],
        "pemalsuan_keywords" : features["pemalsuan_keywords"],
        
        # Teks lengkap (untuk embedding)
        "text_full"          : text
    }
    cases.append(case_record)

print(f"\n✅ {len(cases)} kasus berhasil direpresentasikan")

Memproses kasus: 100%|██████████| 35/35 [00:00<00:00, 52.29it/s]


✅ 35 kasus berhasil direpresentasikan


## 6. Simpan ke CSV dan JSON

In [7]:
df_cases = pd.DataFrame(cases)

# Simpan CSV (tanpa text_full agar tidak terlalu besar)
cols_csv = [
    "case_id", "original_filename", "no_perkara", "tanggal", "jenis_perkara",
    "pasal", "terdakwa", "penuntut", "pengadilan",
    "ringkasan_fakta", "dakwaan", "amar_putusan", "vonis",
    "word_count", "sentence_count", "unique_words",
    "top_keywords", "pemalsuan_keywords"
]
csv_path = PROCESSED_DIR / "cases.csv"
df_cases[cols_csv].to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"✅ CSV disimpan: {csv_path}")

# Simpan JSON lengkap (dengan text_full)
json_path = PROCESSED_DIR / "cases.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(cases, f, ensure_ascii=False, indent=2)
print(f"✅ JSON disimpan: {json_path}")

# Simpan solusi untuk Tahap 4
solutions = {c["case_id"]: c["amar_putusan"] for c in cases}
with open(PROCESSED_DIR / "solutions.json", "w", encoding="utf-8") as f:
    json.dump(solutions, f, ensure_ascii=False, indent=2)
print(f"✅ Solusi disimpan: {PROCESSED_DIR}/solutions.json")

✅ CSV disimpan: ..\data\processed\cases.csv
✅ JSON disimpan: ..\data\processed\cases.json
✅ Solusi disimpan: ..\data\processed/solutions.json


## 7. Analisis & Preview

In [8]:
# Statistik dataset
print("📊 Statistik Dataset:")
print(f"  Total kasus             : {len(df_cases)}")
print(f"  Rata-rata kata/dokumen  : {df_cases['word_count'].mean():.0f}")
print(f"  Nomor perkara ditemukan : {(df_cases['no_perkara'] != 'TIDAK_DITEMUKAN').sum()}")
print(f"  Tanggal ditemukan       : {(df_cases['tanggal'] != 'TIDAK_DITEMUKAN').sum()}")
print(f"  Pasal ditemukan         : {(df_cases['pasal'] != 'TIDAK_DITEMUKAN').sum()}")
print(f"  Terdakwa ditemukan      : {(df_cases['terdakwa'] != 'TIDAK_DITEMUKAN').sum()}")
print(f"  Amar putusan ditemukan  : {(df_cases['amar_putusan'] != 'TIDAK_DITEMUKAN').sum()}")
print(f"  Vonis ditemukan         : {(df_cases['vonis'] != 'TIDAK_DITEMUKAN').sum()}")

print("\n📋 Preview 3 kasus pertama:")
cols_preview = ["case_id", "no_perkara", "tanggal", "pasal", "terdakwa", "vonis"]
print(df_cases[cols_preview].head(3).to_string(index=False))

print("\n✅ Tahap 2 selesai! Lanjutkan ke notebook 03_case_retrieval.ipynb")

📊 Statistik Dataset:
  Total kasus             : 35
  Rata-rata kata/dokumen  : 8662
  Nomor perkara ditemukan : 35
  Tanggal ditemukan       : 35
  Pasal ditemukan         : 34
  Terdakwa ditemukan      : 30
  Amar putusan ditemukan  : 32
  Vonis ditemukan         : 6

📋 Preview 3 kasus pertama:
 case_id             no_perkara    tanggal                          pasal                                                                                                                                                                                                                                                                                                                                         terdakwa           vonis
case_001  309/PID.B/2013/PN.PKL 1969-11-24    374, 253, 263, 263 1, 197 1 Menerangkan Jika Terdakwa Tidak Mempunyani Kewenangan Untuk Meminjamkan Sertifikat Yang Mempuyani Kewenangan Adalah Pimpinan Yaitu Sutiyono Dan Terdakwa Mengatakan Agar Subhan Datang Saja Sendiri Ke Ka